# Numerosity-PRF R² across scanners

Loads the per-(subject, session) R² maps produced by `balgrist/encoding_model/fit_mapper.py` and compares their distributions across the four scanner sessions.

Sessions: `ses-balgrist3t`, `ses-balgrist7t`, `ses-ibt7t`, `ses-sns3t`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns

from balgrist.utils.data import Subject, get_all_subject_ids, SESSIONS

# Set to local mount or cluster path
BIDS_FOLDER = Path('/shares/zne.uzh/gdehol/ds-balgrist')
# BIDS_FOLDER = Path('/Volumes/g_econ_department$/projects/2021/dehollander_nagy_balgristpilots/data/ds-balgrist')

DERIV = 'encoding_model'

## Load R² maps

In [ ]:
def load_r2(subject, session, bids_folder=BIDS_FOLDER, deriv=DERIV):
    """Load a single R² map as a flat (voxel,) array, masked by GM if available."""
    sub = Subject(subject, bids_folder=bids_folder)
    fn = (bids_folder / 'derivatives' / deriv
          / f'sub-{subject}' / f'ses-{session}' / 'func'
          / f'sub-{subject}_ses-{session}_task-mapper_desc-r2_space-T1w_pars.nii.gz')
    if not fn.exists():
        return None
    img = nib.load(str(fn))
    mask = sub.get_conjunct_mask(session).get_fdata() > 0
    data = img.get_fdata()
    return data[mask]

In [ ]:
subjects = get_all_subject_ids(bids_folder=BIDS_FOLDER)
print('subjects:', subjects)

rows = []
for sub in subjects:
    for ses in SESSIONS:
        r2 = load_r2(sub, ses)
        if r2 is None:
            print(f'  missing: sub-{sub} ses-{ses}')
            continue
        rows.append(pd.DataFrame({
            'subject': sub, 'session': ses,
            'field_strength': '7T' if '7t' in ses else '3T',
            'site': ses.replace('3t', '').replace('7t', ''),
            'r2': r2,
        }))

df = pd.concat(rows, ignore_index=True)
df.head()

## Whole-brain R² distribution per session (log-scaled y-axis)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=df[df['r2'] > 0], x='session', y='r2',
               hue='field_strength', ax=ax,
               order=['balgrist3t', 'sns3t', 'balgrist7t', 'ibt7t'],
               inner='quartile', cut=0)
ax.set_yscale('log')
ax.set_ylabel('R² (voxels with R² > 0)')
ax.set_title('Numerosity-PRF R² per scanner session')
plt.tight_layout()

## Tail mass: fraction of voxels above thresholds

In [ ]:
thresholds = [0.01, 0.02, 0.05, 0.10, 0.20]
tail = (df.groupby(['subject', 'session'])['r2']
          .apply(lambda s: pd.Series({t: float((s > t).mean()) for t in thresholds}))
          .reset_index()
          .rename(columns={'level_2': 'threshold', 0: 'frac'}))
tail.head()

In [ ]:
fig, axes = plt.subplots(1, len(thresholds), figsize=(4 * len(thresholds), 4),
                         sharey=True)
for ax, t in zip(axes, thresholds):
    sub = tail[tail['threshold'] == t]
    sns.barplot(data=sub, x='session', y='frac', ax=ax,
                order=['balgrist3t', 'sns3t', 'balgrist7t', 'ibt7t'],
                errorbar='sd')
    ax.set_title(f'R² > {t}')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
axes[0].set_ylabel('fraction of voxels')
plt.tight_layout()

## Paired within-subject 3T vs 7T (Balgrist only)

Compares `ses-balgrist3t` against `ses-balgrist7t` for the same subject — the cleanest within-participant 3T vs 7T contrast since scanner location is matched.

In [ ]:
summary = (df.groupby(['subject', 'session'])
             .agg(median_r2=('r2', 'median'),
                  frac_above_02=('r2', lambda s: float((s > 0.02).mean())),
                  frac_above_10=('r2', lambda s: float((s > 0.10).mean())))
             .reset_index())
summary.pivot(index='subject', columns='session',
              values='frac_above_02').sort_index()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, title in [(axes[0], 'median_r2', 'median R²'),
                        (axes[1], 'frac_above_02', 'fraction R² > 0.02')]:
    wide = summary.pivot(index='subject', columns='session', values=col)
    keep = ['balgrist3t', 'balgrist7t']
    wide = wide[keep].dropna()
    for sub_id, row in wide.iterrows():
        ax.plot(keep, row.values, marker='o', label=f'sub-{sub_id}')
    ax.set_title(title)
    ax.set_xlabel('')
axes[0].legend(fontsize=8, loc='best')
plt.tight_layout()